# Generative AI & Agent Engineering: offline companion notebook
Run in a clean kernel. See SETUP.md. Read corresponding lessons and vary one parameter after each example.


## G02 — structured output
Dependencies: pydantic>=2.


In [ ]:
from pydantic import BaseModel,ConfigDict,Field,ValidationError
class Invoice(BaseModel):
    model_config=ConfigDict(extra='forbid',strict=True)
    currency: str=Field(pattern=r'^[A-Z]{3}$')
    total_minor_units: int=Field(ge=0)
assert Invoice.model_validate_json('{"currency":"INR","total_minor_units":1200}').total_minor_units==1200
try:
    Invoice.model_validate({'currency':'INR','total_minor_units':-1})
except ValidationError: print('invalid amount rejected')
else: raise AssertionError('invalid amount accepted')

## G03 — rag retrieval
Dependencies: numpy scikit-learn.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
# Sparse retrieval baseline; replace with dense embeddings and compare recall.
docs={'leave':'Annual leave is twenty days.', 'travel':'Travel claims require receipts.', 'security':'Never share passwords.'}
vec=TfidfVectorizer(); index=vec.fit_transform(docs.values())
def retrieve(query,k=2):
    scores=cosine_similarity(vec.transform([query]),index)[0]
    return [(list(docs)[i],float(scores[i])) for i in scores.argsort()[::-1][:k] if scores[i]>0]
assert retrieve('travel receipts')[0][0]=='travel'
assert retrieve('spaceship')==[]
print(retrieve('annual leave'))

## G04 — bounded corrective rag
Dependencies: standard library.


In [ ]:
# Offline controller demonstration, not the original CRAG or Self-RAG training algorithm.
def answer(query, retrieve, grade, generate, verify, max_attempts=3):
    for attempt in range(max_attempts):
        evidence=retrieve(query,attempt)
        if not grade(evidence): continue
        draft=generate(query,evidence)
        if verify(draft,evidence): return {'answer':draft,'attempts':attempt+1,'status':'ok'}
    return {'answer':None,'attempts':max_attempts,'status':'abstain'}
result=answer('leave',lambda q,i: [] if i==0 else ['20 days'],bool,lambda q,e:e[0],lambda d,e:d in e)
assert result=={'answer':'20 days','attempts':2,'status':'ok'}
assert answer('unknown',lambda q,i:[],bool,lambda q,e:'',lambda d,e:False)['status']=='abstain'
print(result)

## G05 — tool agent loop
Dependencies: standard library.


In [ ]:
def inventory(item): return {'pencil':12}.get(item,0)
def execute(call):
    if set(call)!={'tool','item'} or call['tool']!='inventory' or not isinstance(call['item'],str): raise ValueError('invalid tool call')
    return inventory(call['item'])
def run(decide,max_steps=4):
    observations=[]
    for _ in range(max_steps):
        action=decide(observations)
        if 'final' in action: return action['final']
        observations.append(execute(action))
    return None
assert run(lambda obs:{'final':obs[-1]} if obs else {'tool':'inventory','item':'pencil'})==12
assert run(lambda obs:{'tool':'inventory','item':'pencil'},2) is None
print('bounded agent tests passed')

## G08 — scoped memory
Dependencies: standard library.


In [ ]:
class Memory:
    def __init__(self): self.records={}
    def put(self,user,key,value): self.records[(user,key)]=value
    def search(self,user,query): return [v for (u,k),v in self.records.items() if u==user and query.lower() in k.lower()]
    def delete(self,user,key): self.records.pop((user,key),None)
m=Memory(); m.put('u1','language','Python'); m.put('u2','language','Java')
assert m.search('u1','language')==['Python']
m.delete('u1','language'); assert not m.search('u1','language')
print('scope and delete checks passed')
# This is a fake store; authenticated identity must come from the runtime.

## G10 — offline eval
Dependencies: standard library.


In [ ]:
def score(case):
    allowed=set(case['retrieved_ids'])
    cited=set(case['cited_ids'])
    return {'citation_precision':len(cited&allowed)/len(cited) if cited else None,'required_tool_ok':case['expected_tool']==case['actual_tool']}
assert score({'retrieved_ids':['A'],'cited_ids':['A','B'],'expected_tool':'search','actual_tool':'search'})=={'citation_precision':.5,'required_tool_ok':True}
print('deterministic evaluation passed')
# Citation membership is not semantic faithfulness; add human-calibrated grading.

## G11 — lora parameter count
Dependencies: torch.


In [ ]:
import torch
from torch import nn
base=nn.Linear(16,8,bias=False); base.requires_grad_(False)
r=2; A=nn.Parameter(torch.randn(r,16)*.01); B=nn.Parameter(torch.zeros(8,r))
x=torch.randn(4,16)
y=base(x)+x@A.T@B.T  # scale alpha/r omitted for this teaching example (equal to 1)
y.square().mean().backward()
assert base.weight.grad is None and B.grad is not None
print('base',base.weight.numel(),'trainable adapter',A.numel()+B.numel())